# Geneva-Copenhagen Survey

We utilize only the first version of the Geneva-Copenhagen Survey (GCS I), focusing on F- and G-type stars.

### Key Reference: Holmberg et al., 2007

* **Paper:** [The Geneva-Copenhagen survey of the Solar neighborhood: Ages, metallicities, and kinematic properties of ~14,000 F and G dwarfs](https://www.aanda.org/articles/aa/abs/2004/18/aa0959/aa0959.html)
* **Data Access (Vizier):** [Geneva-Copenhagen Survey of the Solar neighborhood: V/117A](https://cdsarc.cds.unistra.fr/viz-bin/cat/V/117A#/browse)

To view the parameter descriptions, you can refer to the [ReadMe](https://cdsarc.cds.unistra.fr/ftp/V/117A/ReadMe) file from the Geneva-Copenhagen Survey I (Holmberg et al., 2007).

## Setup

In [ ]:
import os
import numpy as np
import pandas as pd
from astropy.io import fits
from astropy.coordinates import Angle
import warnings

In [ ]:
warnings.simplefilter("ignore", category=fits.verify.VerifyWarning)

In [ ]:
OUTPUT_DIR = "../data/raw/"
DATA_PATH_GCSI = OUTPUT_DIR + "gcs1.fits"

SIMBAD_QUERY_FILE = OUTPUT_DIR + "resources/simbad_query.txt"
STAR_NAMES_FILE = OUTPUT_DIR + "resources/stars_names.txt"

OUTPUT_STARS = "../data/processed/gcs-allstars.csv"
OUTPUT_F_FNAME = "../data/processed/gcs-Fstars.csv"
OUTPUT_G_FNAME = "../data/processed/gcs-Gstars.csv"

## Loading the data

In [ ]:
with fits.open(DATA_PATH_GCSI) as hdul:
    fits_table = hdul[1].data

df = pd.DataFrame.from_records(fits_table)

In [ ]:
total_stars = len(df)
total_parameters = len(df.columns)
stars_with_vsini = len(df[df["vsini"] != 0])
stars_with_mass = len(df[df["mass"] != 0])
mass_min = df["mass"].min()
mass_max = df["mass"].max()

print("## Geneva-Copenhagen Survey Original Data:")
print(f"- Number of stars: {total_stars}")
print(f"- Number of parameters: {total_parameters}")
print(f"- Stars with vsini data: {stars_with_vsini}")
print(f"- Stars with mass data: {stars_with_mass}")
print(f"- Mass ranges from {mass_min} to {mass_max} M⊙\n")

## Convert coordinates

In [ ]:
# Convert RA and DEC from separate columns to single numeric columns in degrees.
def convert_ra(row):
    """
    Convert RA from hours, minutes, seconds to degrees.
    """
    ra_str = f"{int(row['RAh'])} {int(row['RAm'])} {row['RAs']}"
    ra_deg = Angle(ra_str, unit="hourangle").degree
    return ra_deg


def convert_dec(row):
    """
    Convert DEC from degrees, arcminutes, arcseconds to degrees.
    """
    dec_sign = row["DE_"].strip()
    dec_str = f"{dec_sign}{int(row['DEd'])} {int(row['DEm'])} {row['DEs']}"
    dec_deg = Angle(dec_str, unit="degree").degree
    return dec_deg


df["RA"] = df.apply(convert_ra, axis=1)
df["DEC"] = df.apply(convert_dec, axis=1)

## Compute Cartesian coordinates

In [ ]:
# Compute Cartesian coordinates (X, Y, Z) from Galactic longitude and latitude.
# Equations (1) to (3) from our paper
df["X"] = df["Dist"] * np.cos(np.radians(df["GLAT"])) * np.cos(np.radians(df["GLON"]))
df["Y"] = df["Dist"] * np.cos(np.radians(df["GLAT"])) * np.sin(np.radians(df["GLON"]))
df["Z"] = df["Dist"] * np.sin(np.radians(df["GLAT"]))

## Clean data

In [ ]:
# Data cleaning by removing binaries and giants, and ensuring well-defined vsini and distance.
# Flags:
## FB: This flag identifies confirmed and suspected binaries.
# The information can come from one or several sources such as
# photometry, radial velocity or astrometry.

## FD: Flag for spectroscopic binaries.
# Mean radial velocity. For double lined binaries, the
# computed systemic velocity is given if so indicated by the fd flag.

## COMP: If the star is a member of a multiple system the component(s)
# included in the photometry are identified here.

## FG: Flag for suspected giants.
# Indicates a disagreement between the photometric distance
# determination and the Hipparcos parallax at the 3 sigma level,
# suggesting that the star is a giant not detected from the photometry.

# Identify and exclude binaries
binaries_condition = (df["fb"] != "*") & (df["fd"] != "*") & (df["Comp"] == "    ")
df_without_binaries = df[binaries_condition]
binaries_num = len(df) - len(df_without_binaries)
binaries_percent = (binaries_num / len(df)) * 100
print(f"- Number of binary targets: {binaries_num} ({binaries_percent:.2f}%)")

# Identify and exclude suspected giants
giants_condition = df_without_binaries["fg"] != "*"
df_without_giants = df_without_binaries[giants_condition]

# Ensure vsini is well-defined
df_welldefined_vsini = df_without_giants[df_without_giants["vsini"] != 0]

# Ensure distance and Teff are non-zero
df_clean = df_welldefined_vsini[df_welldefined_vsini["Dist"] != 0].reset_index(
    drop=True
)

df_clean = df_welldefined_vsini[df_welldefined_vsini["logTe"] != 0].reset_index(
    drop=True
)

print(f"- Number of suspected giant targets: {len(df[df['fg'] == '*'])}")
print(f"- Number of stars after cleaning: {len(df_clean)}\n")

## Merge SIMBAD data

For the spectral class for each of the sample spectra, we query the identifier in SIMBAD database and check the availability of the stellar classification and object type. According to the [list of object types](https://simbad.cds.unistra.fr/guide/otypes.htx), which is also saved in `data/otypes.list`, we can see that our sample includes the following:

| Object Type | Description                  |
|-------------|------------------------------|
| PM*         | High Proper Motion Star      |
| *           | Star                         |
| SB*         | Spectroscopic Binary         |
| Em*         | Emission-line Star           |
| **          | Double or Multiple Star      |
| BY*         | BY Dra Variable              |
| Er*         | Eruptive Variable            |
| Pe*         | Chemically Peculiar Star     |
| Ro*         | Rotating Variable            |
| RS*         | RS CVn Variable              |
| dS*         | delta Sct Variable           |
| V*          | Variable Star                |
| EB*         | Eclipsing Binary             |
| gD*         | gamma Dor Variable           |
| TT*         | T Tauri Star                 |
| Y*O         | Young Stellar Object         |
| EB?         | Eclipsing Binary             |
| El*         | Ellipsoidal Variable         |
| Pu*         | Pulsating Variable           |
| LM*         | Low-mass Star                |
| HV*         |  High Velocity Star          |
| Ir*         | Irregular Variable           |
| RR?         | RR Lyrae Variable            |
| SB?         | Spectroscopic Binary         |

In [ ]:
# Write star names to a file for SIMBAD querying
with open(STAR_NAMES_FILE, "w") as file:
    for name in df_clean["Name"]:
        file.write(f"{name.strip()}\n")

# Avoiding unnecessary empty spaces
df_clean["Name"] = df_clean["Name"].str.strip()

# Read SIMBAD query results
simbad_df = pd.read_csv(SIMBAD_QUERY_FILE, delimiter="|")

# Read SIMBAD query results
simbad_df = pd.read_csv(SIMBAD_QUERY_FILE, delimiter="|")
simbad_df["typed ident"] = simbad_df["typed ident"].str.strip()

# Clean column names
simbad_df.columns = simbad_df.columns.str.strip()

# Rename columns for clarity
simbad_df = simbad_df.rename(
    columns={"typed ident": "Name", "typ": "obj_type", "spec. type": "spec_type"}
)

# Strip whitespace from relevant columns
simbad_df["obj_type"] = simbad_df["obj_type"].str.strip()
simbad_df["spec_type"] = simbad_df["spec_type"].str.strip()

# Merge with GCS I data
df_merged = df_clean.merge(
    simbad_df[["Name", "obj_type", "spec_type"]], on="Name", how="left"
)

obj_type_counts = df_merged["obj_type"].value_counts().reset_index()
print(f"- SIMBAD object type distribution:")
display(obj_type_counts)

# Based on the previous table, let's filter and keep only the entries
# with object types "PM*" and "*":
valid_obj_types = ["PM*", "*"]
df_filtered = df_merged[df_merged["obj_type"].isin(valid_obj_types)]

## Final datasets

In [ ]:
# Select necessary columns
selected_columns = [
    "Name",
    "fs",
    "RAh",
    "RAm",
    "RAs",
    "DE_",
    "DEd",
    "DEm",
    "DEs",
    "GLON",
    "GLAT",
    "Vmag",
    "b_y",
    "Hbeta",
    "E_b_y_",
    "logTe",
    "_Fe_H_",
    "Dist",
    "VMAG",
    "dVMag",
    "Age",
    "clAge",
    "chAge",
    "mass",
    "clmass",
    "chmass",
    "RVel",
    "meRVel",
    "e_RVel",
    "o_RVel",
    "dT",
    "P_chi2_",
    "vsini",
    "pmRA",
    "pmDE",
    "e_pm",
    "plx",
    "e_plx",
    "UVel",
    "VVel",
    "WVel",
    "Rgal",
    "zgal",
    "Rmin",
    "Rmax",
    "ecc",
    "zmax",
    "RA",
    "DEC",
    "X",
    "Y",
    "Z",
    "obj_type",
    "spec_type",
]

df_final = df_filtered[selected_columns].reset_index(drop=True).copy()
df_final.rename(
    {
        "E_b_y_": "E(b-y)",
        "_Fe_H_": "[Fe/H]",
        "P_chi2_": "P(chi2)",
        "obj_type": "ObjType",
        "spec_type": "SpecType",
    },
    axis=1,
    inplace=True,
)

In [ ]:
# Filter for F and G spectral types
df_final = df_final[df_final["SpecType"].str.startswith(("F", "G"))].reset_index(
    drop=True
)

# Separate into F and G type stars
df_F = df_final[df_final["SpecType"].str.startswith("F")].reset_index(drop=True)
df_G = df_final[df_final["SpecType"].str.startswith("G")].reset_index(drop=True)

print(f"## Final Dataset Summary:")
print(f"- Total number of stars: {len(df_final)}")
print(f"- Number of F-type stars: {len(df_F)}")
print(f"- Number of G-type stars: {len(df_G)}\n")

In [ ]:
df_final.to_csv(OUTPUT_STARS, index=False)
df_F.to_csv(OUTPUT_F_FNAME, index=False)
df_G.to_csv(OUTPUT_G_FNAME, index=False)
print(f"- Exported all type stars to {OUTPUT_STARS}")
print(f"- Exported F-type stars to {OUTPUT_F_FNAME}")
print(f"- Exported G-type stars to {OUTPUT_G_FNAME}")